# 11 – Assessor Attribute Cleaning (Maricopa County)

This notebook cleans and standardizes **assessor attribute data**, such as:

- Parcel ID / APN
- Land use / property classification
- Assessed value, improvement value, land value
- Other relevant fields

**Goals:**
- Load raw assessor table(s) (CSV, DBF, etc.).
- Inspect column names and data types.
- Clean and standardize key fields (especially parcel ID).
- Handle missing values and obvious data issues.
- Save a cleaned attribute table ready to join to parcels.

In [14]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ASSESSOR_DIR = DATA_DIR / "assessor"

ASSESSOR_DIR

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor')

In [15]:
raw_assessor_files = sorted(
    ASSESSOR_DIR.glob("raw/R1170_SecMaster_2024_TAX_ROLL_*.txt")
)
clean_assessor_path = ASSESSOR_DIR / "processed/R1170_SecMaster_2024_TAX_ROLL_clean.csv"

raw_assessor_files, clean_assessor_path

([WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor/raw/R1170_SecMaster_2024_TAX_ROLL_1.txt'),
  WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor/raw/R1170_SecMaster_2024_TAX_ROLL_2.txt'),
  WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor/raw/R1170_SecMaster_2024_TAX_ROLL_3.txt'),
  WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor/raw/R1170_SecMaster_2024_TAX_ROLL_4.txt'),
  WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor/raw/R1170_SecMaster_2024_TAX_ROLL_5.txt'),
  WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor/raw/R1170_SecMaster_2024_TAX_ROLL_6.txt'),
  WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor/raw/R1170_SecMaster_2024_TAX_ROLL_7.txt')],
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/assessor/processed/R1170_SecMaster_2024_TAX_ROLL_clean.csv'))

#### I used Generative AI to extract the layout and had it build a list for column names, since it's 84 columns and I am a lazy individual. I did run through the list myself afterward to make sure AI wasn't hallucinating any names.

In [49]:
COLUMN_NAMES = [
    "PARCEL_NO",            # 1
    "PROPERTYSTATUS",       # 2
    "PROPERTYUSECODE",      # 3
    "TAXAREACODE",          # 4
    "ORGEXEMPTIONCODE",     # 5
    "ORGEXEMPTINDICATOR",   # 6
    "SECTION",              # 7
    "TOWN",                 # 8
    "RANGE",                # 9
    "SECTIONQUARTER",       # 10
    "LOT",                  # 11
    "BLOCK",                # 12
    "TRACT",                # 13
    "SITUSSTREETNUM",       # 14
    "SITUSSTREETDIR",       # 15
    "SITUSSTREETNAME",      # 16
    "SITUSSTREETTYPE",      # 17
    "SITUSSTREETPOSTDIR",   # 18
    "SITUSSUITE",           # 19
    "SITUSCITY",            # 20
    "SITUSZIP",             # 21
    "OWNERSNAME",           # 22
    "OWNERSDEED",           # 23
    "OWNERSDEEDDATE",       # 24
    "FEESNAME",             # 25
    "FEESDEED",             # 26
    "FEESDEEDDATE",         # 27
    "OWNERSADDR1STLINE",    # 28
    "OWNERSADDR2NDLINE",    # 29
    "OWNERSCITY",           # 30
    "OWNERSSTATE",          # 31
    "OWNERSZIPCODE",        # 32
    "OWNERCOUNTRY",         # 33
    "LPVPERCENT",           # 34
    "LPVAMOUNT",            # 35
    "LPVASSESSEDVALUE",     # 36
    "FULLCASHVALUE",        # 37
    "LANDASSESSMENTPERC",   # 38
    "LANDFULLCASHVALUE",    # 39
    "IMPRASSESSMENTPERC",   # 40
    "IMPRFULLCASHVALUE",        # 41
    "WIDVETINDICATOR",          # 42
    "WIDVETLPVEXASSESSEDVALUE", # 43
    "WIDVETFCVEXASSESSEDVALUE", # 44
    "SQFOOTAGE",                # 45
    "VALSOURCE",                # 46
    "OVERRIDECODE",             # 47
    "LANDCLASS1ST",             # 48
    "LANDRATIO1ST",             # 49
    "LAND2ND",                  # 50
    "LANDRATIO2ND",             # 51
    "LAND3RD",                  # 52
    "LANDRATIO3RD",             # 53
    "LAND4TH",                  # 54
    "LANDRATIO4TH",             # 55
    "IMPROVEMENTCLASS1ST",      # 56
    "IMPROVEMENTRATIO1ST",      # 57
    "IMPROVEMENT2ND",           # 58
    "IMPROVEMENTRATIO2ND",      # 59
    "IMPROVEMENT3RD",           # 60
    "IMPROVEMENTRATIO3RD",      # 61
    "IMPROVEMENT4TH",           # 62
    "IMPROVEMENTRATIO4TH",      # 63
    "LANDAREATYPE",             # 64
    "NEIGHBORHOODCODE",         # 65
    "MARKETAREACODE",           # 66
    "MCR",                      # 67
    "SUBDIVISION",              # 68
    "MAILINGDATE",              # 69
    "ECONOMICUNIT",             # 70
    "NUMBEROFUNITS",            # 71
    "DISTRICTCODE1",            # 72
    "DISTRICTCODEVALUE1",       # 73
    "DISTRICTCODE2",            # 74
    "DISTRICTCODEVALUE2",       # 75
    "DISTRICTCODE3",            # 76
    "DISTRICTCODEVALUE3",       # 77
    "DISTRICTCODE4",            # 78
    "DISTRICTCODEVALUE4",       # 79
    "DISTRICTCODE5",            # 80
    "DISTRICTCODEVALUE5",       # 81
    "DISTRICTCODE6",            # 82
    "DISTRICTCODEVALUE6",       # 83
    "LGL_DESC",                 # 84
]


#### Had two different errors below which were debugged with the assistance of AI. Specifically, the encoding and the warning for misalignment (which had previously broken due to the expected extra | somewhere)

In [63]:
dfs = []

for path in raw_assessor_files:
    print(f"Reading {path.name}")
    df = pd.read_csv(
        path,
        sep="|",
        header=None,
        names=COLUMN_NAMES,
        dtype="string",
        engine="python",
        encoding="cp1252",
        index_col=False,
    )
    dfs.append(df)

assessor = pd.concat(dfs, ignore_index=True)
assessor.head()
assessor.info()

Reading R1170_SecMaster_2024_TAX_ROLL_1.txt
Reading R1170_SecMaster_2024_TAX_ROLL_2.txt
Reading R1170_SecMaster_2024_TAX_ROLL_3.txt
Reading R1170_SecMaster_2024_TAX_ROLL_4.txt
Reading R1170_SecMaster_2024_TAX_ROLL_5.txt
Reading R1170_SecMaster_2024_TAX_ROLL_6.txt
Reading R1170_SecMaster_2024_TAX_ROLL_7.txt
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1749565 entries, 0 to 1749564
Data columns (total 84 columns):
 #   Column                    Dtype 
---  ------                    ----- 
 0   PARCEL_NO                 string
 1   PROPERTYSTATUS            string
 2   PROPERTYUSECODE           string
 3   TAXAREACODE               string
 4   ORGEXEMPTIONCODE          string
 5   ORGEXEMPTINDICATOR        string
 6   SECTION                   string
 7   TOWN                      string
 8   RANGE                     string
 9   SECTIONQUARTER            string
 10  LOT                       string
 11  BLOCK                     string
 12  TRACT                     string
 13  SITU

In [64]:
assessor.columns

Index(['PARCEL_NO', 'PROPERTYSTATUS', 'PROPERTYUSECODE', 'TAXAREACODE',
       'ORGEXEMPTIONCODE', 'ORGEXEMPTINDICATOR', 'SECTION', 'TOWN', 'RANGE',
       'SECTIONQUARTER', 'LOT', 'BLOCK', 'TRACT', 'SITUSSTREETNUM',
       'SITUSSTREETDIR', 'SITUSSTREETNAME', 'SITUSSTREETTYPE',
       'SITUSSTREETPOSTDIR', 'SITUSSUITE', 'SITUSCITY', 'SITUSZIP',
       'OWNERSNAME', 'OWNERSDEED', 'OWNERSDEEDDATE', 'FEESNAME', 'FEESDEED',
       'FEESDEEDDATE', 'OWNERSADDR1STLINE', 'OWNERSADDR2NDLINE', 'OWNERSCITY',
       'OWNERSSTATE', 'OWNERSZIPCODE', 'OWNERCOUNTRY', 'LPVPERCENT',
       'LPVAMOUNT', 'LPVASSESSEDVALUE', 'FULLCASHVALUE', 'LANDASSESSMENTPERC',
       'LANDFULLCASHVALUE', 'IMPRASSESSMENTPERC', 'IMPRFULLCASHVALUE',
       'WIDVETINDICATOR', 'WIDVETLPVEXASSESSEDVALUE',
       'WIDVETFCVEXASSESSEDVALUE', 'SQFOOTAGE', 'VALSOURCE', 'OVERRIDECODE',
       'LANDCLASS1ST', 'LANDRATIO1ST', 'LAND2ND', 'LANDRATIO2ND', 'LAND3RD',
       'LANDRATIO3RD', 'LAND4TH', 'LANDRATIO4TH', 'IMPROVEMENTCLASS1S

In [65]:
assessor.dtypes

PARCEL_NO             string[python]
PROPERTYSTATUS        string[python]
PROPERTYUSECODE       string[python]
TAXAREACODE           string[python]
ORGEXEMPTIONCODE      string[python]
                           ...      
DISTRICTCODE5         string[python]
DISTRICTCODEVALUE5    string[python]
DISTRICTCODE6         string[python]
DISTRICTCODEVALUE6    string[python]
LGL_DESC              string[python]
Length: 84, dtype: object

In [71]:
PARCEL_ID_FIELD = "PARCEL_NO"           # Parcel identifier Column 1
LAND_USE_FIELD = "PROPERTYUSECODE"      # Assessor land-use code Column 3

# Value fields for per-acre metrics
TOTAL_VALUE_FIELD = "FULLCASHVALUE"     # Total FCV Column 37
LAND_VALUE_FIELD = "LANDFULLCASHVALUE"  # Land FCV Column 39
IMPROVEMENT_VALUE_FIELD = "IMPRFULLCASHVALUE"  # Improvement FCV Column 41

LPV_FIELD = "LPVAMOUNT"                 # Limited Property Value (taxable) Column 35
ASSESSED_VALUE_FIELD = "LPVASSESSEDVALUE" # Column 36

# Optional but useful
SQFT_FIELD = "SQFOOTAGE"                # Improvement size Column 45
UNITS_FIELD = "NUMBEROFUNITS"           # Residential density Column 71
NEIGHBORHOOD_FIELD = "NEIGHBORHOODCODE"    # Neighborhood models Column 65
MARKETAREA_FIELD = "MARKETAREACODE"        # Market area code for valuation comparisons Column 66

core_cols = [
    PARCEL_ID_FIELD,
    LAND_USE_FIELD,
    TOTAL_VALUE_FIELD,
    LAND_VALUE_FIELD,
    IMPROVEMENT_VALUE_FIELD,
    LPV_FIELD,
    ASSESSED_VALUE_FIELD,
    SQFT_FIELD,
    UNITS_FIELD,
    NEIGHBORHOOD_FIELD,
    MARKETAREA_FIELD,
]


assessor_core = assessor[core_cols].copy()
assessor_core.head()

,PARCEL_NO,PROPERTYUSECODE,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS,NEIGHBORHOODCODE,MARKETAREACODE
0,10101001C,5400,68700,68700,0,68700,11336,7565,<NA>,001,06
1,10101002G,9400,188100,188100,0,89917,13488,63173,<NA>,003,29
2,10101004R,4110,140460,140460,0,84164,12625,3437319,<NA>,001,06
3,10101008B,3799,3996473,3170500,825973,2274012,360329,529389,<NA>,001,06
4,10101009,9705,641500,641500,0,177877,26682,107766,<NA>,001,06


In [72]:
# Standardize parcel IDs: strip whitespace, make uppercase, etc.

assessor_core[PARCEL_ID_FIELD] = (
    assessor_core[PARCEL_ID_FIELD]
    .astype(str)
    .str.strip()
    .str.upper()
)

assessor_core[PARCEL_ID_FIELD].head()

0    10101001C
1    10101002G
2    10101004R
3    10101008B
4     10101009
Name: PARCEL_NO, dtype: object

In [73]:
# Check missing values in key fields
assessor_core.isna().sum()

PARCEL_NO                  0
PROPERTYUSECODE          544
FULLCASHVALUE            546
LANDFULLCASHVALUE        546
IMPRFULLCASHVALUE        546
LPVAMOUNT                545
LPVASSESSEDVALUE       10518
SQFOOTAGE                546
NUMBEROFUNITS        1735907
NEIGHBORHOODCODE         588
MARKETAREACODE           564
dtype: int64

In [74]:
# Example: drop rows with missing parcel IDs
assessor_core = assessor_core[~assessor_core[PARCEL_ID_FIELD].isna()].copy()
print("Rows after dropping missing parcel IDs:", len(assessor_core))

# Example: replace negative or impossible values with NaN
# assessor_core.loc[assessor_core[TOTAL_VALUE_FIELD] < 0, TOTAL_VALUE_FIELD] = pd.NA

Rows after dropping missing parcel IDs: 1749565


In [75]:
assessor_core.to_csv(clean_assessor_path, index=False)
print("Saved cleaned assessor attributes to:", clean_assessor_path)

Saved cleaned assessor attributes to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\assessor\processed\R1170_SecMaster_2024_TAX_ROLL_clean.csv
